### **Installations, Configurations, Imports, Setups and Data Preparation**

In [1]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "torch": "torch",
    "transformers": "transformers>=4.51,<5",
    "datasets": "datasets",
    "pandas": "pandas",
    "numpy": "numpy",
    "sacrebleu": "sacrebleu>=2.4,<3",
    "sentencepiece": "sentencepiece",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm"
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All dependencies are installed.")

print("Python:", sys.version)
print("Executable:", sys.executable)

All dependencies are installed.
Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Executable: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/bin/python


In [2]:
import os
import sys
import gc
import re
import ast
import json
import math
import time
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")

In [3]:
# ============================================================
# EDIT PATHS AND TRAINING PARAMETERS ONLY IN THIS CELL
# Change RUN_NAME whenever starting a materially different run.
# ============================================================

PROJECT_DIR = Path(os.environ.get("AXMT_HOME", str(Path.home() / "alexandriax_mt_14d"))).expanduser()
INFERENCE_VARIANTS_ROOT = PROJECT_DIR / "inference_variants"
SHARED_CACHE_DIR = INFERENCE_VARIANTS_ROOT / "_shared_cache"
PREPARED_CACHE_DIR = SHARED_CACHE_DIR / "paired_dev_train_v3"

RUN_NAME = "94_xlmr_large_dialect_crossencoder_pairrank_v1"
RUN_DIR = PROJECT_DIR / "rerankers" / RUN_NAME
OUTPUT_DIR = INFERENCE_VARIANTS_ROOT / RUN_NAME

LOCAL_ENCODER_DIR = PROJECT_DIR / "models" / "hf" / "xlm-roberta-large"
ENCODER_NAME = str(LOCAL_ENCODER_DIR) if LOCAL_ENCODER_DIR.exists() else "FacebookAI/xlm-roberta-large"

CANDIDATE_VARIANTS = [
    "00_previous_official_control",
    "01_exact_training_parity",
    "02_metadata_no_shots",
    "03_retrieved_two_shot",
    "04_training_parity_with_participants",
    "05_retrieved_two_shot_with_participants",
    "06_ckpt16500_retrieved_two_shot",
    "07_ckpt16000_retrieved_two_shot",
    "08_interp_015_035_050_retrieved_two_shot"
]

BASELINE_VARIANT = "92_mixed_best_checkpoint_variant_per_country"

EXPECTED_DEV_TURNS = 12250
EXPECTED_TRAIN_TURNS = 66480
EXPECTED_DEV_COUNTRIES = 11

# ------------------------------------------------------------
# DIALECT-RESOURCE TRAINING PARAMETERS
# ------------------------------------------------------------

DIALECT_MAX_LENGTH = 160
DIALECT_BATCH_SIZE = 16
DIALECT_GRAD_ACCUM_STEPS = 2
DIALECT_EPOCHS = 2
DIALECT_LR = 1e-5
DIALECT_SAVE_STEPS = 250

# ------------------------------------------------------------
# PAIRWISE RERANKER TRAINING PARAMETERS
# ------------------------------------------------------------

N_FOLDS = 5
RANK_MAX_LENGTH = 384
RANK_BATCH_SIZE = 4
RANK_GRAD_ACCUM_STEPS = 8
RANK_EPOCHS = 2
RANK_ENCODER_LR = 1e-5
RANK_HEAD_LR = 5e-5
RANK_SAVE_STEPS = 250
RANK_TEMPERATURE = 1.0

PAIRS_PER_TURN = 6
MIN_UTILITY_GAP = 0.05
PAIR_WEIGHT_MIN = 0.25
PAIR_WEIGHT_MAX = 4.0

LABEL_SPBLEU_WEIGHT = 0.80
LABEL_CHRF_WEIGHT = 0.20

# ------------------------------------------------------------
# SHARED OPTIMIZATION AND INFERENCE PARAMETERS
# ------------------------------------------------------------

WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

SCORE_BATCH_SIZE = 32
SCORE_SAVE_ROWS = 128
NUM_WORKERS = 0

# 0.0 means only exact score ties fall back to System92.
MIN_SCORE_MARGIN = 0.0

# Package the reranker only when its OOF spBLEU beats System92.
DEPLOY_MIN_SPBLEU_GAIN = 0.0

DIALECT_NAMES = {
    "EG": "Egyptian Arabic",
    "JO": "Jordanian Arabic",
    "LB": "Lebanese Arabic",
    "MA": "Moroccan Arabic",
    "MR": "Mauritanian Arabic",
    "OM": "Omani Arabic",
    "PS": "Palestinian Arabic",
    "SA": "Saudi Arabic",
    "SY": "Syrian Arabic",
    "TN": "Tunisian Arabic",
    "YE": "Yemeni Arabic"
}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for XLM-R-large training.")

USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
DEVICE = torch.device("cuda")

RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def short_hash(value):
    payload = json.dumps(value, sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()[:16]

DIALECT_SIGNATURE = short_hash({
    "encoder": ENCODER_NAME,
    "max_length": DIALECT_MAX_LENGTH,
    "batch": DIALECT_BATCH_SIZE,
    "accum": DIALECT_GRAD_ACCUM_STEPS,
    "epochs": DIALECT_EPOCHS,
    "lr": DIALECT_LR,
    "seed": SEED
})

RANK_SIGNATURE = short_hash({
    "dialect": DIALECT_SIGNATURE,
    "variants": CANDIDATE_VARIANTS,
    "folds": N_FOLDS,
    "max_length": RANK_MAX_LENGTH,
    "batch": RANK_BATCH_SIZE,
    "accum": RANK_GRAD_ACCUM_STEPS,
    "epochs": RANK_EPOCHS,
    "encoder_lr": RANK_ENCODER_LR,
    "head_lr": RANK_HEAD_LR,
    "pairs": PAIRS_PER_TURN,
    "min_gap": MIN_UTILITY_GAP,
    "label_weights": [LABEL_SPBLEU_WEIGHT, LABEL_CHRF_WEIGHT],
    "seed": SEED
})

print("Run:", RUN_NAME)
print("Encoder:", ENCODER_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Training dtype:", AMP_DTYPE)
print("Run directory:", RUN_DIR)
print("Output directory:", OUTPUT_DIR)

Run: 94_xlmr_large_dialect_crossencoder_pairrank_v1
Encoder: FacebookAI/xlm-roberta-large
GPU: NVIDIA GeForce RTX 5090
Training dtype: torch.bfloat16
Run directory: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1
Output directory: /home/mabdallah/alexandriax_mt_14d/inference_variants/94_xlmr_large_dialect_crossencoder_pairrank_v1


In [4]:
DEV_CACHE_PATH = PREPARED_CACHE_DIR / "official_dev_df.pkl"
TRAIN_CACHE_PATH = PREPARED_CACHE_DIR / "train_fewshot_pool_df.pkl"

if not DEV_CACHE_PATH.exists() or not TRAIN_CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Prepared cache missing under {PREPARED_CACHE_DIR}. "
        "Run the data-preparation cell in Inference_Variants first."
    )

official_dev_df = pd.read_pickle(DEV_CACHE_PATH)
official_dev_df = official_dev_df.sort_values(
    ["config", "conversation_id", "turn_order"]
).reset_index(drop=True)

train_gold_df = pd.read_pickle(TRAIN_CACHE_PATH).reset_index(drop=True)

for column in ["source_id", "config", "conversation_id", "source_text", "reference_arabic"]:
    official_dev_df[column] = official_dev_df[column].fillna("").astype(str)

for column in ["config", "conversation_id", "target_arabic"]:
    train_gold_df[column] = train_gold_df[column].fillna("").astype(str)

official_dev_df["turn_order"] = pd.to_numeric(
    official_dev_df["turn_order"], errors="raise"
).astype(int)

if len(official_dev_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(f"Expected {EXPECTED_DEV_TURNS} DEV turns, found {len(official_dev_df)}.")

if len(train_gold_df) != EXPECTED_TRAIN_TURNS:
    raise RuntimeError(f"Expected {EXPECTED_TRAIN_TURNS} train turns, found {len(train_gold_df)}.")

if official_dev_df["config"].nunique() != EXPECTED_DEV_COUNTRIES:
    raise RuntimeError("Unexpected number of DEV countries.")

def locate_prediction_csv(folder):
    for filename in ["turn_predictions.csv", "scored_turn_predictions.csv"]:
        path = folder / filename
        if path.exists():
            return path
    raise FileNotFoundError(f"No turn prediction CSV found under {folder}")

def load_ordered_prediction(folder, keep_selected=False):
    raw = pd.read_csv(locate_prediction_csv(folder))

    if "source_id" not in raw or "prediction" not in raw:
        raise ValueError(f"Bad prediction schema: {folder}")

    raw["source_id"] = raw["source_id"].astype(str)
    raw["prediction"] = raw["prediction"].fillna("").astype(str).str.strip()

    if raw["source_id"].duplicated().any() or (raw["prediction"] == "").any():
        raise RuntimeError(f"Duplicate or empty prediction in {folder}")

    columns = ["source_id", "prediction"]

    if keep_selected and "selected_from_variant" in raw:
        columns.append("selected_from_variant")

    ordered = official_dev_df[["source_id"]].merge(
        raw[columns], on="source_id", how="left", validate="one_to_one"
    )

    if ordered["prediction"].isna().any():
        raise RuntimeError(f"Incomplete predictions in {folder}")

    return ordered

variant_names = list(CANDIDATE_VARIANTS)

candidate_frames = [
    load_ordered_prediction(INFERENCE_VARIANTS_ROOT / name)
    for name in variant_names
]

candidate_texts = np.column_stack([
    frame["prediction"].to_numpy(dtype=object)
    for frame in candidate_frames
])

baseline_frame = load_ordered_prediction(
    INFERENCE_VARIANTS_ROOT / BASELINE_VARIANT,
    keep_selected=True
)

baseline_predictions = baseline_frame["prediction"].to_numpy(dtype=object)

def normalized_text(value):
    return re.sub(r"\s+", " ", str(value)).strip()

variant_to_idx = {name: index for index, name in enumerate(variant_names)}
baseline_idx = np.full(len(official_dev_df), -1, dtype=np.int16)

for row_index, baseline_text in enumerate(baseline_predictions):
    selected_name = ""

    if "selected_from_variant" in baseline_frame:
        selected_name = str(
            baseline_frame.loc[row_index, "selected_from_variant"]
        ).strip()

    candidate_index = variant_to_idx.get(selected_name, -1)

    if (
        candidate_index < 0
        or normalized_text(candidate_texts[row_index, candidate_index])
        != normalized_text(baseline_text)
    ):
        matches = [
            index
            for index in range(len(variant_names))
            if normalized_text(candidate_texts[row_index, index])
            == normalized_text(baseline_text)
        ]

        candidate_index = matches[0] if matches else -1

    if candidate_index < 0:
        source_id = official_dev_df.loc[row_index, "source_id"]
        raise RuntimeError(
            f"System92 prediction is absent from the candidate pool: {source_id}"
        )

    baseline_idx[row_index] = candidate_index

candidate_hasher = hashlib.sha256()

for name in variant_names:
    candidate_hasher.update(name.encode())

for text in candidate_texts.reshape(-1):
    candidate_hasher.update(str(text).encode("utf-8"))
    candidate_hasher.update(b"\0")

CANDIDATE_HASH = candidate_hasher.hexdigest()[:20]
N, K = candidate_texts.shape

print("DEV rows:", len(official_dev_df))
print("Gold training rows:", len(train_gold_df))
print("Candidates per turn:", K)
print("Candidate variants:", variant_names)
print("Candidate hash:", CANDIDATE_HASH)

DEV rows: 12250
Gold training rows: 66480
Candidates per turn: 9
Candidate variants: ['00_previous_official_control', '01_exact_training_parity', '02_metadata_no_shots', '03_retrieved_two_shot', '04_training_parity_with_participants', '05_retrieved_two_shot_with_participants', '06_ckpt16500_retrieved_two_shot', '07_ckpt16000_retrieved_two_shot', '08_interp_015_035_050_retrieved_two_shot']
Candidate hash: 4442f3c1d2e6d6dfaeb3


### **Create conversation-grouped OOF folds**

In [5]:
groups = (
    official_dev_df["config"]
    + "::"
    + official_dev_df["conversation_id"]
)

fold_id = np.full(N, -1, dtype=np.int8)

splitter = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

for fold, (_, validation_indices) in enumerate(
    splitter.split(
        np.zeros(N),
        official_dev_df["config"],
        groups
    )
):
    fold_id[validation_indices] = fold

if (fold_id < 0).any():
    raise RuntimeError("Fold assignment is incomplete.")

fold_check = (
    official_dev_df
    .assign(fold=fold_id)
    .groupby(["config", "conversation_id"])["fold"]
    .nunique()
    .max()
)

if fold_check != 1:
    raise RuntimeError("Conversation leakage across folds.")

display(pd.crosstab(
    official_dev_df["config"],
    fold_id,
    margins=True
))

col_0,0,1,2,3,4,All
config,,,,,,
EG,224,223,222,222,222,1113
JO,223,224,222,222,222,1113
LB,223,223,223,225,224,1118
MA,223,221,222,222,222,1110
MR,222,222,224,224,222,1114
OM,222,221,221,222,223,1109
PS,222,223,221,221,223,1110
SA,222,221,221,223,223,1110
SY,223,225,225,223,223,1119


### **Define the models**

In [6]:
def configure_encoder(encoder):
    encoder.config.use_cache = False

    if hasattr(encoder, "gradient_checkpointing_enable"):
        encoder.gradient_checkpointing_enable()

    return encoder

class DialectClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.encoder = configure_encoder(
            AutoModel.from_pretrained(model_name)
        )

        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.10)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0]

        return self.classifier(self.dropout(hidden))

class DialectAwareQualityModel(nn.Module):
    def __init__(self, model_name, dialect_feature_size=3):
        super().__init__()

        self.encoder = configure_encoder(
            AutoModel.from_pretrained(model_name)
        )

        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size + dialect_feature_size),
            nn.Linear(hidden_size + dialect_feature_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, input_ids, attention_mask, dialect_features):
        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0]

        combined = torch.cat([hidden, dialect_features], dim=-1)
        return self.head(combined).squeeze(-1)

### **Checkpoint and mixed-precision helpers**

In [7]:
def amp_context():
    return torch.autocast("cuda", dtype=AMP_DTYPE)

def new_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda", enabled=not USE_BF16
        )
    except TypeError:
        return torch.cuda.amp.GradScaler(
            enabled=not USE_BF16
        )

def atomic_torch_save(obj, path):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temporary_path)
    os.replace(temporary_path, path)

def atomic_numpy_save(array, path):
    path = Path(path)
    temporary_path = path.with_suffix(".tmp.npy")
    np.save(temporary_path, array)
    os.replace(temporary_path, path)

def atomic_npz_save(path, **arrays):
    path = Path(path)
    temporary_path = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary_path, **arrays)
    os.replace(temporary_path, path)

def atomic_json_save(obj, path):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(obj, file, ensure_ascii=False, indent=2)

    os.replace(temporary_path, path)

def load_torch(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu"
        )

def optimizer_to(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

def capture_rng():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all()
    }

def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    torch.cuda.set_rng_state_all(state["cuda"])

tokenizer = AutoTokenizer.from_pretrained(
    ENCODER_NAME,
    use_fast=True
)

tokenizer.save_pretrained(RUN_DIR / "tokenizer")

print("Tokenizer ready.")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


### **Prepare the 66,480-example dialect resource**

In [8]:
dialect_codes = sorted(
    train_gold_df["config"].unique().tolist()
)

dialect_to_id = {
    code: index
    for index, code in enumerate(dialect_codes)
}

if not set(official_dev_df["config"]).issubset(dialect_to_id):
    raise RuntimeError(
        "A DEV dialect has no gold training class."
    )

dialect_texts = (
    train_gold_df["target_arabic"]
    .str.strip()
    .tolist()
)

dialect_labels = (
    train_gold_df["config"]
    .map(dialect_to_id)
    .to_numpy(np.int64)
)

class_counts = np.bincount(
    dialect_labels,
    minlength=len(dialect_codes)
)

class_weights = np.sqrt(
    class_counts.mean() / class_counts
).astype(np.float32)

class DialectDataset(Dataset):
    def __len__(self):
        return len(dialect_texts)

    def __getitem__(self, index):
        return (
            dialect_texts[index],
            int(dialect_labels[index])
        )

class DialectCollator:
    def __call__(self, items):
        texts, labels = zip(*items)

        batch = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=DIALECT_MAX_LENGTH,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor(
            labels,
            dtype=torch.long
        )

        return batch

print("Dialect classes:")
print(dict(zip(
    dialect_codes,
    class_counts.tolist()
)))

Dialect classes:
{'EG': 3108, 'JO': 5501, 'LB': 8906, 'MA': 2573, 'MR': 5515, 'OM': 6280, 'PS': 14933, 'SA': 8470, 'SY': 6071, 'TN': 2034, 'YE': 3089}


### **Train or resume the dialect classifier**

In [9]:
DIALECT_DIR = RUN_DIR / "dialect_resource"
DIALECT_DIR.mkdir(parents=True, exist_ok=True)

DIALECT_LAST_PATH = DIALECT_DIR / "checkpoint_last.pt"
DIALECT_FINAL_PATH = DIALECT_DIR / "model_final.pt"

def save_dialect_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    next_batch,
    step
):
    atomic_torch_save({
        "signature": DIALECT_SIGNATURE,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "next_batch": next_batch,
        "step": step,
        "rng": capture_rng(),
        "dialect_codes": dialect_codes
    }, path)

def train_dialect_resource():
    model = DialectClassifier(
        ENCODER_NAME,
        len(dialect_codes)
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=DIALECT_LR,
        weight_decay=WEIGHT_DECAY
    )

    batches_per_epoch = math.ceil(
        len(dialect_texts) / DIALECT_BATCH_SIZE
    )

    updates_per_epoch = math.ceil(
        batches_per_epoch
        / DIALECT_GRAD_ACCUM_STEPS
    )

    total_updates = (
        updates_per_epoch
        * DIALECT_EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_updates * WARMUP_RATIO),
        total_updates
    )

    scaler = new_scaler()
    start_epoch = 0
    start_batch = 0
    global_step = 0

    if DIALECT_FINAL_PATH.exists():
        state = load_torch(DIALECT_FINAL_PATH)

        if state["signature"] != DIALECT_SIGNATURE:
            raise RuntimeError(
                "Dialect final checkpoint conflicts with "
                "current parameters. Change RUN_NAME."
            )

        model.load_state_dict(state["model"])

        print(
            "Dialect resource already complete:",
            DIALECT_FINAL_PATH
        )

        return model

    if DIALECT_LAST_PATH.exists():
        state = load_torch(DIALECT_LAST_PATH)

        if state["signature"] != DIALECT_SIGNATURE:
            raise RuntimeError(
                "Dialect resume checkpoint conflicts with "
                "current parameters. Change RUN_NAME."
            )

        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        optimizer_to(optimizer, DEVICE)
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["step"]

        restore_rng(state["rng"])

        print(
            f"Resuming dialect training at "
            f"epoch={start_epoch}, "
            f"batch={start_batch}, "
            f"step={global_step}"
        )

    weights = torch.tensor(
        class_weights,
        device=DEVICE
    )

    for epoch in range(
        start_epoch,
        DIALECT_EPOCHS
    ):
        generator = torch.Generator()
        generator.manual_seed(SEED + epoch)

        loader = DataLoader(
            DialectDataset(),
            batch_size=DIALECT_BATCH_SIZE,
            shuffle=True,
            generator=generator,
            collate_fn=DialectCollator(),
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=(
                f"Dialect epoch "
                f"{epoch + 1}/{DIALECT_EPOCHS}"
            )
        )

        for batch_index, batch in progress:
            if (
                epoch == start_epoch
                and batch_index < start_batch
            ):
                continue

            labels = batch.pop("labels").to(
                DEVICE,
                non_blocking=True
            )

            inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True
                )
                for key, value in batch.items()
            }

            with amp_context():
                logits = model(**inputs)

                loss = F.cross_entropy(
                    logits,
                    labels,
                    weight=weights
                )

            scaled_loss = (
                loss
                / DIALECT_GRAD_ACCUM_STEPS
            )

            scaler.scale(
                scaled_loss
            ).backward()

            running_loss += float(
                loss.detach()
            )

            do_step = (
                (batch_index + 1)
                % DIALECT_GRAD_ACCUM_STEPS
                == 0
                or batch_index + 1 == len(loader)
            )

            if do_step:
                scaler.unscale_(optimizer)

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                progress.set_postfix(
                    loss=(
                        f"{running_loss / max(1, batch_index + 1):.4f}"
                    ),
                    step=global_step
                )

                if (
                    global_step
                    % DIALECT_SAVE_STEPS
                    == 0
                ):
                    save_dialect_checkpoint(
                        DIALECT_LAST_PATH,
                        model,
                        optimizer,
                        scheduler,
                        scaler,
                        epoch,
                        batch_index + 1,
                        global_step
                    )

        start_batch = 0

        save_dialect_checkpoint(
            DIALECT_LAST_PATH,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch + 1,
            0,
            global_step
        )

    atomic_torch_save({
        "signature": DIALECT_SIGNATURE,
        "model": model.state_dict(),
        "dialect_codes": dialect_codes,
        "step": global_step
    }, DIALECT_FINAL_PATH)

    print(
        "Saved dialect resource:",
        DIALECT_FINAL_PATH
    )

    return model

dialect_model = train_dialect_resource()

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Dialect epoch 1/2:   0%|          | 0/4155 [00:00<?, ?it/s]

Dialect epoch 2/2:   0%|          | 0/4155 [00:00<?, ?it/s]

Saved dialect resource: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/dialect_resource/model_final.pt


### **Score every candidate for requested-dialect compatibility**

In [10]:
DIALECT_FEATURE_PATH = (
    RUN_DIR
    / "candidate_dialect_features.npz"
)

def compute_dialect_features(model):
    if DIALECT_FEATURE_PATH.exists():
        cache = np.load(
            DIALECT_FEATURE_PATH,
            allow_pickle=False
        )

        if (
            cache["signature"].item()
            == DIALECT_SIGNATURE
            and cache["candidate_hash"].item()
            == CANDIDATE_HASH
        ):
            print(
                "Loaded dialect feature cache:",
                DIALECT_FEATURE_PATH
            )

            return cache["features"]

    model.eval()

    features = np.empty(
        (N * K, 3),
        dtype=np.float32
    )

    flat_texts = (
        candidate_texts
        .reshape(-1)
        .tolist()
    )

    target_ids = np.repeat(
        official_dev_df["config"]
        .map(dialect_to_id)
        .to_numpy(np.int64),
        K
    )

    for start in tqdm(
        range(
            0,
            len(flat_texts),
            SCORE_BATCH_SIZE
        ),
        desc="Candidate dialect scores"
    ):
        end = min(
            start + SCORE_BATCH_SIZE,
            len(flat_texts)
        )

        batch = tokenizer(
            flat_texts[start:end],
            padding=True,
            truncation=True,
            max_length=DIALECT_MAX_LENGTH,
            return_tensors="pt"
        )

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.inference_mode(), amp_context():
            logits = model(**batch).float()

        log_probabilities = logits.log_softmax(-1)
        probabilities = log_probabilities.exp()

        ids = torch.tensor(
            target_ids[start:end],
            device=DEVICE
        )

        row_indices = torch.arange(
            end - start,
            device=DEVICE
        )

        target_log_probability = (
            log_probabilities[
                row_indices,
                ids
            ]
        )

        other_log_probabilities = (
            log_probabilities.clone()
        )

        other_log_probabilities[
            row_indices,
            ids
        ] = -torch.inf

        target_probability = probabilities[
            row_indices,
            ids
        ]

        margin = (
            target_log_probability
            - other_log_probabilities
            .max(-1)
            .values
        ).clamp(-10, 10) / 10

        entropy = -(
            probabilities
            * log_probabilities
        ).sum(-1)

        certainty = (
            1
            - entropy / math.log(
                len(dialect_codes)
            )
        ).clamp(0, 1)

        features[start:end] = torch.stack([
            target_probability.float(),
            margin.float(),
            certainty.float()
        ], dim=-1).cpu().numpy()

    features = features.reshape(
        N,
        K,
        3
    )

    atomic_npz_save(
        DIALECT_FEATURE_PATH,
        features=features,
        signature=np.array(
            DIALECT_SIGNATURE
        ),
        candidate_hash=np.array(
            CANDIDATE_HASH
        )
    )

    print(
        "Saved dialect feature cache:",
        DIALECT_FEATURE_PATH
    )

    return features

dialect_features = compute_dialect_features(
    dialect_model
)

del dialect_model
gc.collect()
torch.cuda.empty_cache()

dialect_diagnostic = pd.DataFrame({
    "variant": variant_names,
    "mean_requested_dialect_probability": (
        dialect_features[:, :, 0]
        .mean(0)
    )
}).sort_values(
    "mean_requested_dialect_probability",
    ascending=False
)

display(dialect_diagnostic)

Candidate dialect scores:   0%|          | 0/3446 [00:00<?, ?it/s]

Saved dialect feature cache: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/candidate_dialect_features.npz


,variant,mean_requested_dialect_probability
3,03_retrieved_two_shot,0.712610
5,05_retrieved_two_shot_with_participants,0.710283
1,01_exact_training_parity,0.709211
4,04_training_parity_with_participants,0.707199
6,06_ckpt16500_retrieved_two_shot,0.704373
7,07_ckpt16000_retrieved_two_shot,0.696466
2,02_metadata_no_shots,0.695586
0,00_previous_official_control,0.694489
8,08_interp_015_035_050_retrieved_two_shot,0.690959


### **Build exact reference-derived training statistics**

In [11]:
LABEL_CACHE_PATH = (
    RUN_DIR
    / "candidate_metric_statistics.npz"
)

bleu_metric = BLEU(
    tokenize="flores200"
)

chrf_metric = CHRF(
    word_order=2
)

def build_metric_cache():
    if LABEL_CACHE_PATH.exists():
        cache = np.load(
            LABEL_CACHE_PATH,
            allow_pickle=False
        )

        if (
            cache["candidate_hash"].item()
            == CANDIDATE_HASH
        ):
            print(
                "Loaded metric-statistics cache:",
                LABEL_CACHE_PATH
            )

            return (
                cache["bleu_stats"],
                cache["chrf_scores"]
            )

    references = official_dev_df[
        "reference_arabic"
    ].tolist()

    bleu_stats = np.empty(
        (N, K, 10),
        dtype=np.int64
    )

    chrf_scores = np.empty(
        (N, K),
        dtype=np.float32
    )

    for candidate_index, name in enumerate(
        tqdm(
            variant_names,
            desc="Reference-derived label cache"
        )
    ):
        hypotheses = candidate_texts[
            :,
            candidate_index
        ].tolist()

        candidate_bleu_stats = (
            bleu_metric
            ._extract_corpus_statistics(
                hypotheses,
                [references]
            )
        )

        candidate_chrf_stats = (
            chrf_metric
            ._extract_corpus_statistics(
                hypotheses,
                [references]
            )
        )

        bleu_stats[
            :,
            candidate_index
        ] = np.asarray(
            candidate_bleu_stats,
            dtype=np.int64
        )

        chrf_scores[
            :,
            candidate_index
        ] = np.asarray([
            chrf_metric
            ._compute_score_from_stats([
                int(value)
                for value in statistics
            ])
            .score
            for statistics
            in candidate_chrf_stats
        ], dtype=np.float32)

    atomic_npz_save(
        LABEL_CACHE_PATH,
        bleu_stats=bleu_stats,
        chrf_scores=chrf_scores,
        candidate_hash=np.array(
            CANDIDATE_HASH
        )
    )

    print(
        "Saved metric-statistics cache:",
        LABEL_CACHE_PATH
    )

    return bleu_stats, chrf_scores

bleu_stats, chrf_scores = build_metric_cache()

baseline_bleu_stats = bleu_stats[
    np.arange(N),
    baseline_idx
]

baseline_chrf_scores = chrf_scores[
    np.arange(N),
    baseline_idx
]

print("Metric statistics ready.")

Reference-derived label cache:   0%|          | 0/9 [00:00<?, ?it/s]

Saved metric-statistics cache: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/candidate_metric_statistics.npz
Metric statistics ready.


### **Build fold-specific winner–loser pairs**

In [12]:
def bleu_from_stats(statistics):
    score = bleu_metric._compute_score_from_stats([
        int(value)
        for value in statistics
    ])

    return float(score.score)

def robust_z(values):
    values = np.asarray(
        values,
        dtype=np.float64
    )

    median = np.median(values)

    scale = 1.4826 * np.median(
        np.abs(values - median)
    )

    if scale < 1e-12:
        scale = values.std()

    if scale < 1e-12:
        return np.zeros_like(values)

    return (
        values - median
    ) / scale

def build_fold_pairs(fold):
    fold_dir = RUN_DIR / f"fold_{fold}"
    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    path = (
        fold_dir
        / "training_pairs.npz"
    )

    if path.exists():
        cache = np.load(
            path,
            allow_pickle=False
        )

        if (
            cache["signature"].item()
            == RANK_SIGNATURE
            and cache["candidate_hash"].item()
            == CANDIDATE_HASH
        ):
            print(
                f"Fold {fold}: loaded "
                f"{len(cache['row_idx']):,} "
                "cached pairs"
            )

            return {
                key: cache[key]
                for key in [
                    "row_idx",
                    "winner_idx",
                    "loser_idx",
                    "weight"
                ]
            }

    training_indices = np.where(
        fold_id != fold
    )[0]

    utility = np.full(
        (N, K),
        np.nan,
        dtype=np.float32
    )

    configurations = official_dev_df[
        "config"
    ].to_numpy()

    for country in sorted(
        set(configurations[training_indices])
    ):
        country_indices = training_indices[
            configurations[training_indices]
            == country
        ]

        baseline_sum = (
            baseline_bleu_stats[
                country_indices
            ]
            .sum(0)
        )

        baseline_score = bleu_from_stats(
            baseline_sum
        )

        marginal_spbleu = np.empty(
            (
                len(country_indices),
                K
            ),
            dtype=np.float64
        )

        for local_index, row_index in enumerate(
            country_indices
        ):
            for candidate_index in range(K):
                replacement_statistics = (
                    baseline_sum
                    - baseline_bleu_stats[
                        row_index
                    ]
                    + bleu_stats[
                        row_index,
                        candidate_index
                    ]
                )

                marginal_spbleu[
                    local_index,
                    candidate_index
                ] = (
                    bleu_from_stats(
                        replacement_statistics
                    )
                    - baseline_score
                )

        chrf_delta = (
            chrf_scores[country_indices]
            - baseline_chrf_scores[
                country_indices,
                None
            ]
        )

        normalized_spbleu = robust_z(
            marginal_spbleu.reshape(-1)
        ).reshape(
            marginal_spbleu.shape
        )

        normalized_chrf = robust_z(
            chrf_delta.reshape(-1)
        ).reshape(
            chrf_delta.shape
        )

        utility[country_indices] = (
            LABEL_SPBLEU_WEIGHT
            * normalized_spbleu
            + LABEL_CHRF_WEIGHT
            * normalized_chrf
        )

    rows = []
    winners = []
    losers = []
    weights = []

    hard_count = (
        PAIRS_PER_TURN // 2
    )

    for row_index in tqdm(
        training_indices,
        desc=f"Fold {fold} pairs"
    ):
        unique_candidates = {}

        for candidate_index in range(K):
            key = normalized_text(
                candidate_texts[
                    row_index,
                    candidate_index
                ]
            )

            unique_candidates.setdefault(
                key,
                candidate_index
            )

        indices = list(
            unique_candidates.values()
        )

        available_pairs = []

        for left in range(len(indices)):
            for right in range(
                left + 1,
                len(indices)
            ):
                first = indices[left]
                second = indices[right]

                gap = abs(float(
                    utility[row_index, first]
                    - utility[row_index, second]
                ))

                if gap < MIN_UTILITY_GAP:
                    continue

                if (
                    utility[row_index, first]
                    > utility[row_index, second]
                ):
                    winner = first
                    loser = second
                else:
                    winner = second
                    loser = first

                available_pairs.append((
                    gap,
                    winner,
                    loser
                ))

        ordered_pairs = sorted(
            available_pairs
        )

        chosen_pairs = ordered_pairs[
            :hard_count
        ]

        used = {
            (winner, loser)
            for _, winner, loser
            in chosen_pairs
        }

        for item in reversed(
            ordered_pairs
        ):
            if (
                len(chosen_pairs)
                >= PAIRS_PER_TURN
            ):
                break

            pair_key = (
                item[1],
                item[2]
            )

            if pair_key not in used:
                chosen_pairs.append(item)
                used.add(pair_key)

        for gap, winner, loser in chosen_pairs:
            rows.append(row_index)
            winners.append(winner)
            losers.append(loser)

            weights.append(np.clip(
                gap,
                PAIR_WEIGHT_MIN,
                PAIR_WEIGHT_MAX
            ))

    pairs = {
        "row_idx": np.asarray(
            rows,
            dtype=np.int32
        ),
        "winner_idx": np.asarray(
            winners,
            dtype=np.int8
        ),
        "loser_idx": np.asarray(
            losers,
            dtype=np.int8
        ),
        "weight": np.asarray(
            weights,
            dtype=np.float32
        )
    }

    atomic_npz_save(
        path,
        **pairs,
        signature=np.array(
            RANK_SIGNATURE
        ),
        candidate_hash=np.array(
            CANDIDATE_HASH
        )
    )

    print(
        f"Fold {fold}: saved "
        f"{len(rows):,} pairs"
    )

    return pairs

### **Construct source-aware quality inputs and batches**

In [13]:
DEV_ROWS = official_dev_df.to_dict(
    "records"
)

def context_string(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except Exception:
            value = []

    if not isinstance(value, list):
        return "None"

    parts = []

    for turn in value:
        if not isinstance(turn, dict):
            continue

        speaker = str(
            turn.get("speaker", "")
        ).strip()

        direction = str(
            turn.get("direction", "")
        ).strip()

        text = str(
            turn.get("text", "")
        ).strip()

        prefix = "/".join(
            item
            for item in [
                speaker,
                direction
            ]
            if item
        )

        if text:
            parts.append(
                f"{prefix}: {text}"
                if prefix
                else text
            )

    return (
        " <turn> ".join(parts)
        if parts
        else "None"
    )

def quality_input(row, candidate):
    code = str(row["config"])
    dialect = DIALECT_NAMES.get(
        code,
        code
    )

    return (
        f"Requested dialect: {code} ({dialect})\n"
        f"Domain: {row.get('domain', '')}\n"
        f"Speaker: {row.get('speaker', '')}\n"
        f"Direction: {row.get('gender_direction', '')}\n"
        f"Previous English turns: "
        f"{context_string(row.get('previous_english_turns', []))}\n"
        f"Current English source: {row['source_text']}\n"
        f"Arabic candidate: {candidate}"
    )

class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(
            self.pairs["row_idx"]
        )

    def __getitem__(self, index):
        return (
            int(
                self.pairs[
                    "row_idx"
                ][index]
            ),
            int(
                self.pairs[
                    "winner_idx"
                ][index]
            ),
            int(
                self.pairs[
                    "loser_idx"
                ][index]
            ),
            float(
                self.pairs[
                    "weight"
                ][index]
            )
        )

class PairCollator:
    def __call__(self, items):
        rows, winners, losers, weights = zip(
            *items
        )

        winner_texts = [
            quality_input(
                DEV_ROWS[row],
                candidate_texts[
                    row,
                    winner
                ]
            )
            for row, winner
            in zip(rows, winners)
        ]

        loser_texts = [
            quality_input(
                DEV_ROWS[row],
                candidate_texts[
                    row,
                    loser
                ]
            )
            for row, loser
            in zip(rows, losers)
        ]

        winner_features = np.stack([
            dialect_features[
                row,
                winner
            ]
            for row, winner
            in zip(rows, winners)
        ])

        loser_features = np.stack([
            dialect_features[
                row,
                loser
            ]
            for row, loser
            in zip(rows, losers)
        ])

        features = np.vstack([
            winner_features,
            loser_features
        ])

        batch = tokenizer(
            winner_texts + loser_texts,
            padding=True,
            truncation=True,
            max_length=RANK_MAX_LENGTH,
            return_tensors="pt"
        )

        batch["dialect_features"] = torch.tensor(
            features,
            dtype=torch.float32
        )

        batch["weights"] = torch.tensor(
            weights,
            dtype=torch.float32
        )

        batch["pair_batch_size"] = len(items)

        return batch

### **Define resumable fold training and held-out scoring**

In [14]:
def encoder_state_from_dialect_checkpoint():
    state = load_torch(
        DIALECT_FINAL_PATH
    )

    if (
        state["signature"]
        != DIALECT_SIGNATURE
    ):
        raise RuntimeError(
            "Dialect checkpoint signature mismatch."
        )

    return {
        key[len("encoder."):]: value
        for key, value
        in state["model"].items()
        if key.startswith("encoder.")
    }

DIALECT_ENCODER_STATE = (
    encoder_state_from_dialect_checkpoint()
)

def save_rank_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    next_batch,
    step
):
    atomic_torch_save({
        "signature": RANK_SIGNATURE,
        "candidate_hash": CANDIDATE_HASH,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "next_batch": next_batch,
        "step": step,
        "rng": capture_rng()
    }, path)

def train_rank_fold(fold, pairs):
    fold_dir = (
        RUN_DIR
        / f"fold_{fold}"
    )

    last_path = (
        fold_dir
        / "checkpoint_last.pt"
    )

    final_path = (
        fold_dir
        / "model_final.pt"
    )

    model = DialectAwareQualityModel(
        ENCODER_NAME
    ).to(DEVICE)

    incompatible = (
        model.encoder
        .load_state_dict(
            DIALECT_ENCODER_STATE,
            strict=True
        )
    )

    if (
        incompatible.missing_keys
        or incompatible.unexpected_keys
    ):
        raise RuntimeError(
            str(incompatible)
        )

    optimizer = torch.optim.AdamW([
        {
            "params": model.encoder.parameters(),
            "lr": RANK_ENCODER_LR
        },
        {
            "params": model.head.parameters(),
            "lr": RANK_HEAD_LR
        }
    ], weight_decay=WEIGHT_DECAY)

    batches_per_epoch = math.ceil(
        len(pairs["row_idx"])
        / RANK_BATCH_SIZE
    )

    updates_per_epoch = math.ceil(
        batches_per_epoch
        / RANK_GRAD_ACCUM_STEPS
    )

    total_updates = (
        updates_per_epoch
        * RANK_EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_updates * WARMUP_RATIO),
        total_updates
    )

    scaler = new_scaler()
    start_epoch = 0
    start_batch = 0
    global_step = 0

    if final_path.exists():
        state = load_torch(final_path)

        if (
            state["signature"]
            != RANK_SIGNATURE
            or state["candidate_hash"]
            != CANDIDATE_HASH
        ):
            raise RuntimeError(
                f"Fold {fold} final checkpoint "
                "conflicts with the current run. "
                "Change RUN_NAME."
            )

        model.load_state_dict(
            state["model"]
        )

        print(
            f"Fold {fold}: "
            "training already complete"
        )

        return model

    if last_path.exists():
        state = load_torch(last_path)

        if (
            state["signature"]
            != RANK_SIGNATURE
            or state["candidate_hash"]
            != CANDIDATE_HASH
        ):
            raise RuntimeError(
                f"Fold {fold} resume checkpoint "
                "conflicts with the current run. "
                "Change RUN_NAME."
            )

        model.load_state_dict(
            state["model"]
        )

        optimizer.load_state_dict(
            state["optimizer"]
        )

        optimizer_to(
            optimizer,
            DEVICE
        )

        scheduler.load_state_dict(
            state["scheduler"]
        )

        scaler.load_state_dict(
            state["scaler"]
        )

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["step"]

        restore_rng(state["rng"])

        print(
            f"Fold {fold}: resume "
            f"epoch={start_epoch}, "
            f"batch={start_batch}, "
            f"step={global_step}"
        )

    dataset = PairDataset(pairs)

    for epoch in range(
        start_epoch,
        RANK_EPOCHS
    ):
        generator = torch.Generator()

        generator.manual_seed(
            SEED
            + 1000 * fold
            + epoch
        )

        loader = DataLoader(
            dataset,
            batch_size=RANK_BATCH_SIZE,
            shuffle=True,
            generator=generator,
            collate_fn=PairCollator(),
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=(
                f"Fold {fold} rank epoch "
                f"{epoch + 1}/{RANK_EPOCHS}"
            )
        )

        for batch_index, batch in progress:
            if (
                epoch == start_epoch
                and batch_index < start_batch
            ):
                continue

            pair_batch_size = batch.pop(
                "pair_batch_size"
            )

            weights = batch.pop(
                "weights"
            ).to(DEVICE)

            inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True
                )
                for key, value
                in batch.items()
            }

            with amp_context():
                scores = model(**inputs)

                winner_scores = scores[
                    :pair_batch_size
                ]

                loser_scores = scores[
                    pair_batch_size:
                ]

                pair_losses = F.softplus(
                    -(
                        winner_scores
                        - loser_scores
                    )
                    / RANK_TEMPERATURE
                )

                loss = (
                    pair_losses
                    * weights
                ).sum() / weights.sum()

            scaled_loss = (
                loss
                / RANK_GRAD_ACCUM_STEPS
            )

            scaler.scale(
                scaled_loss
            ).backward()

            running_loss += float(
                loss.detach()
            )

            do_step = (
                (batch_index + 1)
                % RANK_GRAD_ACCUM_STEPS
                == 0
                or batch_index + 1
                == len(loader)
            )

            if do_step:
                scaler.unscale_(optimizer)

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                progress.set_postfix(
                    loss=(
                        f"{running_loss / max(1, batch_index + 1):.4f}"
                    ),
                    step=global_step
                )

                if (
                    global_step
                    % RANK_SAVE_STEPS
                    == 0
                ):
                    save_rank_checkpoint(
                        last_path,
                        model,
                        optimizer,
                        scheduler,
                        scaler,
                        epoch,
                        batch_index + 1,
                        global_step
                    )

        start_batch = 0

        save_rank_checkpoint(
            last_path,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch + 1,
            0,
            global_step
        )

    atomic_torch_save({
        "signature": RANK_SIGNATURE,
        "candidate_hash": CANDIDATE_HASH,
        "model": model.state_dict(),
        "step": global_step
    }, final_path)

    print(
        f"Fold {fold}: saved final model to "
        f"{final_path}"
    )

    return model

def score_fold(fold, model):
    fold_dir = (
        RUN_DIR
        / f"fold_{fold}"
    )

    score_path = (
        fold_dir
        / "heldout_candidate_scores.npy"
    )

    metadata_path = (
        fold_dir
        / "heldout_candidate_scores.json"
    )

    validation_indices = np.where(
        fold_id == fold
    )[0]

    expected_ids = official_dev_df.loc[
        validation_indices,
        "source_id"
    ].tolist()

    if (
        score_path.exists()
        and metadata_path.exists()
    ):
        metadata = json.loads(
            metadata_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            metadata["signature"]
            != RANK_SIGNATURE
            or metadata["candidate_hash"]
            != CANDIDATE_HASH
            or metadata["source_ids"]
            != expected_ids
        ):
            raise RuntimeError(
                f"Fold {fold} score cache mismatch."
            )

        scores = np.load(score_path)

        if scores.shape != (
            len(validation_indices),
            K
        ):
            raise RuntimeError(
                f"Fold {fold} score-cache "
                "shape mismatch."
            )

    else:
        scores = np.full(
            (
                len(validation_indices),
                K
            ),
            np.nan,
            dtype=np.float32
        )

        atomic_json_save({
            "signature": RANK_SIGNATURE,
            "candidate_hash": CANDIDATE_HASH,
            "source_ids": expected_ids
        }, metadata_path)

    pending = np.where(
        ~np.isfinite(scores).all(1)
    )[0]

    model.eval()

    for start in tqdm(
        range(
            0,
            len(pending),
            SCORE_SAVE_ROWS
        ),
        desc=f"Fold {fold} held-out scoring"
    ):
        local_positions = pending[
            start:start + SCORE_SAVE_ROWS
        ]

        row_indices = validation_indices[
            local_positions
        ]

        texts = [
            quality_input(
                DEV_ROWS[row_index],
                candidate_texts[
                    row_index,
                    candidate_index
                ]
            )
            for row_index in row_indices
            for candidate_index in range(K)
        ]

        features = np.vstack([
            dialect_features[
                row_index,
                candidate_index
            ]
            for row_index in row_indices
            for candidate_index in range(K)
        ])

        chunk_scores = []

        for batch_start in range(
            0,
            len(texts),
            SCORE_BATCH_SIZE
        ):
            batch_end = min(
                batch_start + SCORE_BATCH_SIZE,
                len(texts)
            )

            batch = tokenizer(
                texts[
                    batch_start:batch_end
                ],
                padding=True,
                truncation=True,
                max_length=RANK_MAX_LENGTH,
                return_tensors="pt"
            )

            batch = {
                key: value.to(DEVICE)
                for key, value
                in batch.items()
            }

            feature_batch = torch.tensor(
                features[
                    batch_start:batch_end
                ],
                dtype=torch.float32,
                device=DEVICE
            )

            with torch.inference_mode(), amp_context():
                batch_scores = model(
                    **batch,
                    dialect_features=feature_batch
                ).float().cpu().numpy()

            chunk_scores.append(
                batch_scores
            )

        scores[local_positions] = (
            np.concatenate(
                chunk_scores
            )
            .reshape(
                len(local_positions),
                K
            )
        )

        atomic_numpy_save(
            scores,
            score_path
        )

    if not np.isfinite(scores).all():
        raise RuntimeError(
            f"Fold {fold} scoring is incomplete."
        )

    return validation_indices, scores

### **Run or resume all five OOF folds**

In [15]:
oof_scores = np.full(
    (N, K),
    np.nan,
    dtype=np.float32
)

for fold in range(N_FOLDS):
    print("\n" + "=" * 90)
    print(
        f"OUTER FOLD "
        f"{fold + 1}/{N_FOLDS}"
    )
    print("=" * 90)

    fold_pairs = build_fold_pairs(
        fold
    )

    rank_model = train_rank_fold(
        fold,
        fold_pairs
    )

    validation_indices, validation_scores = (
        score_fold(
            fold,
            rank_model
        )
    )

    oof_scores[
        validation_indices
    ] = validation_scores

    del rank_model
    del fold_pairs

    gc.collect()
    torch.cuda.empty_cache()

if not np.isfinite(oof_scores).all():
    raise RuntimeError(
        "OOF score matrix is incomplete."
    )

OOF_SCORE_PATH = (
    OUTPUT_DIR
    / "oof_candidate_scores.npy"
)

atomic_numpy_save(
    oof_scores,
    OOF_SCORE_PATH
)

print(
    "Saved complete OOF score matrix:",
    OOF_SCORE_PATH
)


OUTER FOLD 1/5


Fold 0 pairs:   0%|          | 0/9799 [00:00<?, ?it/s]

Fold 0: saved 48,174 pairs


Fold 0 rank epoch 1/2:   0%|          | 0/12044 [00:00<?, ?it/s]

Fold 0 rank epoch 2/2:   0%|          | 0/12044 [00:00<?, ?it/s]

...

Fold 4 rank epoch 1/2:   0%|          | 0/12043 [00:00<?, ?it/s]

Fold 4 rank epoch 2/2:   0%|          | 0/12043 [00:00<?, ?it/s]

Fold 4: saved final model to /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/fold_4/model_final.pt


Fold 4 held-out scoring:   0%|          | 0/20 [00:00<?, ?it/s]

Saved complete OOF score matrix: /home/mabdallah/alexandriax_mt_14d/inference_variants/94_xlmr_large_dialect_crossencoder_pairrank_v1/oof_candidate_scores.npy


### **Select the highest-scoring candidate**

In [16]:
best_idx = oof_scores.argmax(1)

sorted_scores = np.sort(
    oof_scores,
    axis=1
)

score_margin = (
    sorted_scores[:, -1]
    - sorted_scores[:, -2]
)

fallback = (
    score_margin
    <= MIN_SCORE_MARGIN
)

selected_idx = np.where(
    fallback,
    baseline_idx,
    best_idx
)

selected_predictions = candidate_texts[
    np.arange(N),
    selected_idx
]

selected_variants = np.asarray(
    variant_names,
    dtype=object
)[selected_idx]

baseline_variants = np.asarray(
    variant_names,
    dtype=object
)[baseline_idx]

reranker_turn_df = official_dev_df[[
    "source_id",
    "config",
    "country",
    "conversation_id",
    "turn_order",
    "source_text"
]].copy()

reranker_turn_df["prediction"] = (
    selected_predictions
)

reranker_turn_df[
    "selected_from_variant"
] = selected_variants

reranker_turn_df[
    "selected_candidate_index"
] = selected_idx

reranker_turn_df[
    "selected_score"
] = oof_scores[
    np.arange(N),
    selected_idx
]

reranker_turn_df[
    "runner_up_margin"
] = score_margin

reranker_turn_df[
    "requested_dialect_probability"
] = dialect_features[
    np.arange(N),
    selected_idx,
    0
]

reranker_turn_df[
    "baseline_from_variant"
] = baseline_variants

reranker_turn_df[
    "baseline_candidate_index"
] = baseline_idx

reranker_turn_df[
    "changed_from_system92"
] = [
    normalized_text(selected)
    != normalized_text(baseline)
    for selected, baseline
    in zip(
        selected_predictions,
        baseline_predictions
    )
]

RERANKER_TURN_PATH = (
    OUTPUT_DIR
    / "reranker_oof_turn_predictions.csv"
)

reranker_turn_df.to_csv(
    RERANKER_TURN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Text-level overrides:",
    int(
        reranker_turn_df[
            "changed_from_system92"
        ].sum()
    ),
    "/",
    N
)

display(pd.crosstab(
    reranker_turn_df["config"],
    reranker_turn_df[
        "selected_from_variant"
    ],
    margins=True
))

Text-level overrides: 5859 / 12250


selected_from_variant,00_previous_official_control,01_exact_training_parity,02_metadata_no_shots,03_retrieved_two_shot,04_training_parity_with_participants,05_retrieved_two_shot_with_participants,06_ckpt16500_retrieved_two_shot,07_ckpt16000_retrieved_two_shot,08_interp_015_035_050_retrieved_two_shot,All
config,,,,,,,,,,
EG,67,30,29,29,20,545,66,157,170,1113
JO,67,13,34,549,24,29,65,167,165,1113
LB,60,36,25,27,30,29,60,688,163,1118
MA,68,38,47,38,45,509,78,153,134,1110
MR,74,38,59,40,41,39,78,580,165,1114
OM,72,34,25,30,25,499,71,177,176,1109
PS,62,25,39,23,34,23,586,167,151,1110
SA,66,32,24,24,20,25,591,159,169,1110
SY,50,35,26,30,20,580,60,145,173,1119


### **Calculate official DEV metrics and choose the deployable system**

In [17]:
def official_metrics(
    predictions,
    system_name
):
    scored = official_dev_df[[
        "source_id",
        "config",
        "country",
        "conversation_id",
        "turn_order",
        "source_text",
        "reference_arabic"
    ]].copy()

    scored["prediction"] = np.asarray(
        predictions,
        dtype=object
    )

    rows = []

    for country in sorted(
        scored["config"].unique()
    ):
        country_df = scored[
            scored["config"] == country
        ]

        hypotheses = country_df[
            "prediction"
        ].astype(str).tolist()

        references = country_df[
            "reference_arabic"
        ].astype(str).tolist()

        rows.append({
            "country": country,
            "turns": len(country_df),
            "spBLEU": sacrebleu.corpus_bleu(
                hypotheses,
                [references],
                tokenize="flores200"
            ).score,
            "chrF++": sacrebleu.corpus_chrf(
                hypotheses,
                [references],
                word_order=2
            ).score,
            "BLEU": sacrebleu.corpus_bleu(
                hypotheses,
                [references]
            ).score,
            "chrF": sacrebleu.corpus_chrf(
                hypotheses,
                [references],
                word_order=0
            ).score
        })

    per_country = pd.DataFrame(rows)

    summary = {
        "system": system_name,
        "Average spBLEU (primary)": float(
            per_country["spBLEU"].mean()
        ),
        "Average chrF++": float(
            per_country["chrF++"].mean()
        )
    }

    return (
        summary,
        per_country,
        scored
    )

reranker_summary, reranker_per_country, reranker_scored = official_metrics(
    selected_predictions,
    RUN_NAME + "_OOF"
)

baseline_summary, baseline_per_country, baseline_scored = official_metrics(
    baseline_predictions,
    BASELINE_VARIANT
)

comparison = pd.DataFrame([
    baseline_summary,
    reranker_summary
])

display(comparison)

country_comparison = baseline_per_country.merge(
    reranker_per_country,
    on=["country", "turns"],
    suffixes=(
        "_system92",
        "_reranker"
    )
)

country_comparison["spBLEU_delta"] = (
    country_comparison["spBLEU_reranker"]
    - country_comparison["spBLEU_system92"]
)

country_comparison["chrF++_delta"] = (
    country_comparison["chrF++_reranker"]
    - country_comparison["chrF++_system92"]
)

display(country_comparison)

reranker_gain = (
    reranker_summary[
        "Average spBLEU (primary)"
    ]
    - baseline_summary[
        "Average spBLEU (primary)"
    ]
)

deploy_reranker = (
    reranker_gain
    > DEPLOY_MIN_SPBLEU_GAIN
)

final_system = (
    RUN_NAME + "_OOF"
    if deploy_reranker
    else BASELINE_VARIANT
)

final_predictions = (
    selected_predictions
    if deploy_reranker
    else baseline_predictions
)

final_summary, final_per_country, final_scored = official_metrics(
    final_predictions,
    final_system
)

comparison.to_csv(
    OUTPUT_DIR
    / "dev_system_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

country_comparison.to_csv(
    OUTPUT_DIR
    / "dev_per_country_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

atomic_json_save({
    "reranker_gain_spBLEU": reranker_gain,
    "minimum_required_gain": DEPLOY_MIN_SPBLEU_GAIN,
    "deployed_system": final_system,
    "reranker_summary": reranker_summary,
    "baseline_summary": baseline_summary
}, OUTPUT_DIR / "deployment_decision.json")

print(
    f"Deployment decision: {final_system}"
)

print(
    f"OOF spBLEU delta: "
    f"{reranker_gain:+.6f}"
)

,system,Average spBLEU (primary),Average chrF++
0,92_mixed_best_checkpoint_variant_per_country,30.928003,45.594526
1,94_xlmr_large_dialect_crossencoder_pairrank_v1...,30.222754,44.962056


,country,turns,spBLEU_system92,chrF++_system92,BLEU_system92,chrF_system92,spBLEU_reranker,chrF++_reranker,BLEU_reranker,chrF_reranker,spBLEU_delta,chrF++_delta
0,EG,1113,32.903189,46.892198,19.510451,49.894903,32.738883,46.739184,19.110522,49.734969,-0.164306,-0.153014
1,JO,1113,35.301943,49.377895,19.819243,52.901916,34.668512,48.677459,19.263121,52.214888,-0.633431,-0.700435
2,LB,1118,31.701457,45.871215,20.021014,48.586333,31.052344,45.219550,19.293188,47.921522,-0.649113,-0.651664
3,MA,1110,23.530143,39.376601,12.865149,42.236961,23.159985,38.816883,12.617613,41.573243,-0.370158,-0.559718
4,MR,1114,17.380438,33.733013,7.100699,37.779153,16.726189,33.026090,6.780756,37.056715,-0.654249,-0.706923
5,OM,1109,36.147408,49.843410,20.805106,53.247687,34.835102,48.923516,19.794308,52.396910,-1.312306,-0.919894
6,PS,1110,33.418402,47.762506,20.699325,51.031341,32.469488,47.081747,20.157544,50.296290,-0.948914,-0.680759
7,SA,1110,33.166094,48.135685,18.903768,52.048449,32.289077,47.363709,18.123875,51.260292,-0.877017,-0.771976
8,SY,1119,39.993111,53.985574,26.598272,56.940070,39.368395,53.414321,26.245524,56.364856,-0.624716,-0.571253
9,TN,1116,29.285225,43.452865,17.062352,46.059624,28.730792,43.025277,16.524239,45.630695,-0.554433,-0.427588


Deployment decision: 92_mixed_best_checkpoint_variant_per_country
OOF spBLEU delta: -0.705249


### **Write official files and submission ZIP**

In [18]:
submission_turn_df = (
    reranker_turn_df.copy()
)

if not deploy_reranker:
    submission_turn_df[
        "prediction"
    ] = baseline_predictions

    submission_turn_df[
        "selected_from_variant"
    ] = baseline_variants

    submission_turn_df[
        "selected_candidate_index"
    ] = baseline_idx

submission_turn_df.to_csv(
    OUTPUT_DIR
    / "turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

final_scored.to_csv(
    OUTPUT_DIR
    / "scored_turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

final_per_country.to_csv(
    OUTPUT_DIR
    / "per_country_official_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

score_row = {
    "Variant": final_system,
    "Checkpoint": "5-fold OOF",
    **{
        key: value
        for key, value
        in final_summary.items()
        if key != "system"
    }
}

for row in final_per_country.to_dict(
    "records"
):
    score_row[
        f"{row['country']} spBLEU"
    ] = row["spBLEU"]

    score_row[
        f"{row['country']} chrF++"
    ] = row["chrF++"]

pd.DataFrame([
    score_row
]).to_csv(
    OUTPUT_DIR
    / "official_leaderboard_score_row.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics = {
    "variant_name": final_system,
    "evaluation": (
        "conversation-grouped "
        "5-fold OOF"
    ),
    "num_turns": N,
    "num_countries": EXPECTED_DEV_COUNTRIES,
    **{
        key: value
        for key, value
        in final_summary.items()
        if key != "system"
    },
    "per_country": (
        final_per_country
        .to_dict("records")
    ),
    "reranker_spBLEU_gain_over_system92": (
        reranker_gain
    ),
    "candidate_variants": variant_names,
    "encoder": ENCODER_NAME,
    "sacrebleu_version": sacrebleu.__version__,
    "spbleu_tokenizer": "flores200",
    "chrf_word_order": 2
}

atomic_json_save(
    metrics,
    OUTPUT_DIR
    / "official_metrics.json"
)

submission_records = []

ordered = final_scored.sort_values([
    "config",
    "conversation_id",
    "turn_order"
])

for (
    country,
    conversation_id
), conversation_df in ordered.groupby(
    ["config", "conversation_id"],
    sort=True
):
    if conversation_df[
        "turn_order"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate turn order in "
            f"{country}/{conversation_id}"
        )

    turns = [
        {
            "turn_order": int(
                row.turn_order
            ),
            "prediction": str(
                row.prediction
            )
        }
        for row
        in conversation_df.itertuples()
    ]

    submission_records.append({
        "conv_id": str(
            conversation_id
        ),
        "country": str(country),
        "turns": turns
    })

jsonl_path = (
    OUTPUT_DIR
    / "predictions.jsonl"
)

with open(
    jsonl_path,
    "w",
    encoding="utf-8"
) as file:
    for record in submission_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

zip_path = (
    OUTPUT_DIR
    / "submission_predictions.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:
    zip_file.write(
        jsonl_path,
        arcname="predictions.jsonl"
    )

readback_keys = set()
readback_turns = 0

with zipfile.ZipFile(
    zip_path
) as zip_file:
    if zip_file.namelist() != [
        "predictions.jsonl"
    ]:
        raise RuntimeError(
            "ZIP must contain only "
            "predictions.jsonl"
        )

    with zip_file.open(
        "predictions.jsonl"
    ) as file:
        for line in file:
            record = json.loads(
                line.decode("utf-8")
            )

            for turn in record["turns"]:
                readback_turns += 1

                readback_keys.add((
                    str(record["country"]),
                    str(record["conv_id"]),
                    int(turn["turn_order"])
                ))

expected_keys = set(zip(
    official_dev_df["config"],
    official_dev_df[
        "conversation_id"
    ],
    official_dev_df[
        "turn_order"
    ]
))

if (
    readback_turns
    != EXPECTED_DEV_TURNS
    or readback_keys
    != expected_keys
):
    raise RuntimeError(
        "Submission ZIP readback "
        "validation failed."
    )

print("\nOFFICIAL-STYLE DEVELOPMENT RESULT")
print("System:", final_system)

print(
    "Average spBLEU:",
    f"{final_summary['Average spBLEU (primary)']:.6f}"
)

print(
    "Average chrF++:",
    f"{final_summary['Average chrF++']:.6f}"
)

print("\nSUBMIT THIS ZIP:")
print(zip_path)

display(final_per_country)


OFFICIAL-STYLE DEVELOPMENT RESULT
System: 92_mixed_best_checkpoint_variant_per_country
Average spBLEU: 30.928003
Average chrF++: 45.594526

SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/94_xlmr_large_dialect_crossencoder_pairrank_v1/submission_predictions.zip


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.903189,46.892198,19.510451,49.894903
1,JO,1113,35.301943,49.377895,19.819243,52.901916
2,LB,1118,31.701457,45.871215,20.021014,48.586333
3,MA,1110,23.530143,39.376601,12.865149,42.236961
4,MR,1114,17.380438,33.733013,7.100699,37.779153
5,OM,1109,36.147408,49.843410,20.805106,53.247687
6,PS,1110,33.418402,47.762506,20.699325,51.031341
7,SA,1110,33.166094,48.135685,18.903768,52.048449
8,SY,1119,39.993111,53.985574,26.598272,56.940070
9,TN,1116,29.285225,43.452865,17.062352,46.059624
